In [ ]:
# Example 1: Public API surface — validators, formats, and exported names.
import openapi_schema_validator

public_names = [n for n in dir(openapi_schema_validator) if not n.startswith('_')]
print(f"Public names ({len(public_names)}):")
for n in sorted(public_names):
    print(f"  {n}: {type(getattr(openapi_schema_validator, n)).__name__}")

# Core validator classes should be present
from openapi_schema_validator import OAS30Validator, OAS31Validator

assert OAS30Validator is not None
assert OAS31Validator is not None
print(f"\nOAS30Validator: {OAS30Validator}")
print(f"OAS31Validator: {OAS31Validator}")

In [ ]:
# Example 2: Validate a valid OpenAPI 3.0 schema instance with OAS30Validator.
from openapi_schema_validator import OAS30Validator

schema = {
    "type": "object",
    "required": ["name", "age"],
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0},
        "email": {"type": "string", "format": "email"},
    },
}

instance = {"name": "Alice", "age": 30, "email": "alice@example.com"}
validator = OAS30Validator(schema)
errors = list(validator.iter_errors(instance))
print(f"Validation errors for a valid instance: {len(errors)}")
assert errors == []

# validate() raises on failure, returns None on success
result = validator.validate(instance)
assert result is None
print("OAS30Validator.validate() passed")

In [ ]:
# Example 3: Detect invalid instances — missing required field + wrong type.
from openapi_schema_validator import OAS30Validator
from jsonschema.exceptions import ValidationError

schema = {
    "type": "object",
    "required": ["name", "age"],
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer", "minimum": 0},
    },
}

bad_instance = {"name": "Bob", "age": -5}  # negative age violates minimum
validator = OAS30Validator(schema)
errors = list(validator.iter_errors(bad_instance))
print(f"Errors for bad_instance: {len(errors)}")
for err in errors:
    print(f"  - {err.message}")
assert len(errors) >= 1

missing_required = {"name": "Carol"}
try:
    validator.validate(missing_required)
    raise AssertionError("Expected ValidationError")
except ValidationError as e:
    print(f"\nCaught expected ValidationError: {e.message}")

In [ ]:
# Example 4: OpenAPI 3.1 validator with the nullable-via-array-type pattern.
from openapi_schema_validator import OAS31Validator

# OAS 3.1 uses JSON Schema 2020-12 — `type: ["string", "null"]` works directly
schema = {
    "type": "object",
    "properties": {
        "nickname": {"type": ["string", "null"]},
        "score": {"type": "number"},
    },
}

v = OAS31Validator(schema)

for inst in [{"nickname": "Dee", "score": 9.5}, {"nickname": None, "score": 0}]:
    errs = list(v.iter_errors(inst))
    print(f"{inst}: {len(errs)} errors")
    assert errs == []

# Wrong type still rejected
bad_errs = list(v.iter_errors({"nickname": 42, "score": 1.0}))
print(f"\nbad instance errors: {len(bad_errs)}")
assert len(bad_errs) >= 1

In [ ]:
# Example 5: Format checker — OpenAPI-specific formats (date, date-time, uuid).
from openapi_schema_validator import OAS30Validator, oas30_format_checker

schema = {
    "type": "object",
    "properties": {
        "created": {"type": "string", "format": "date-time"},
        "id": {"type": "string", "format": "uuid"},
    },
}

validator = OAS30Validator(schema, format_checker=oas30_format_checker)

good = {"created": "2024-01-15T10:30:00Z", "id": "550e8400-e29b-41d4-a716-446655440000"}
good_errors = list(validator.iter_errors(good))
print(f"Good instance: {len(good_errors)} errors")
assert good_errors == []

bad = {"created": "not-a-date", "id": "not-a-uuid"}
bad_errors = list(validator.iter_errors(bad))
print(f"Bad instance: {len(bad_errors)} errors")
for e in bad_errors:
    print(f"  - {e.message}")
assert len(bad_errors) >= 1, "format_checker should reject malformed date-time/uuid"